<a href="https://colab.research.google.com/github/pink-heart199/Avishka-Gaykar/blob/main/TASK_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎨 Conditional Image Colorization
### Internship Project — DeOldify + Semantic Segmentation + User-Defined Color Conditions

**Description:** This notebook colorizes grayscale images using DeOldify as the base engine,  
then applies user-defined color conditions to specific semantic regions (sky, grass, water, etc.)  
via a fully interactive Gradio GUI.

---
**Stack:** Python 3.11 · PyTorch · DeOldify · SegFormer · OpenCV · NumPy · Gradio  
**Runtime:** GPU (T4 recommended)

---
## SECTION 1 — Install Dependencies

In [ ]:
# ============================================================
# SECTION 1 — Install Dependencies
# ============================================================
# Install all required packages for the project.
# DeOldify is cloned from source; all other deps via pip.

import subprocess, sys

def run(cmd):
    """Run a shell command and stream output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[WARN] Command returned non-zero: {cmd}")
        print(result.stderr[-500:] if result.stderr else "")
    else:
        print(f"[OK] {cmd.split()[0]}")

print("Step 1/6 — Upgrading pip...")
run("pip install --quiet --upgrade pip")

print("Step 2/6 — Installing core ML libraries...")
run("pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

!pip install yt-dlp

import torch
print(torch.__version__)

print("Step 3/6 — Installing image processing & UI libraries...")
run(
    "pip install --quiet "
    "opencv-python-headless "
    "Pillow "
    "numpy "
    "matplotlib "
    "scipy "
    "scikit-image "
    "ffmpeg-python "          # ← ADD THIS
    "gradio>=4.0.0 "
    "transformers>=4.35.0 "
    "accelerate "
    "timm "
    "fastai==2.7.13 "
    "fastcore "
    "requests "
    "tqdm "
    "colorama"
)
run("apt-get install -qq ffmpeg")  # system binary
run("pip install --quiet ffmpeg-python")  # Python binding

print("Step 4/6 — Cloning DeOldify...")
import os
if not os.path.exists("/content/DeOldify"):
    run("git clone --quiet https://github.com/jantic/DeOldify.git /content/DeOldify")
else:
    print("[OK] DeOldify already cloned.")

print("Step 5/6 — Creating output directories...")
os.makedirs("/content/DeOldify/models", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)
os.makedirs("/content/uploads", exist_ok=True)

print("Step 6/6 — Downloading DeOldify Artistic model weights...")
MODEL_PATH = "/content/DeOldify/models/ColorizeArtistic_gen.pth"
if not os.path.exists(MODEL_PATH):
    run(
        'wget --quiet -O /content/DeOldify/models/ColorizeArtistic_gen.pth '
        '"https://data.deepai.org/deoldify/ColorizeArtistic_gen.pth"'
    )
    # Fallback mirror
    if not os.path.exists(MODEL_PATH) or os.path.getsize(MODEL_PATH) < 1_000_000:
        print("[INFO] Primary mirror failed. Trying HuggingFace mirror...")
        run(
            'wget --quiet -O /content/DeOldify/models/ColorizeArtistic_gen.pth '
            '"https://huggingface.co/spandanmadan/deoldify/resolve/main/ColorizeArtistic_gen.pth"'
        )
else:
    print("[OK] Model weights already present.")

print("\n✅ All dependencies installed successfully!")

Step 1/6 — Upgrading pip...
[OK] pip
Step 2/6 — Installing core ML libraries...
[OK] pip
2.5.1+cu124
Step 3/6 — Installing image processing & UI libraries...
[WARN] Command returned non-zero: pip install --quiet opencv-python-headless Pillow numpy matplotlib scipy scikit-image ffmpeg-python gradio>=4.0.0 transformers>=4.35.0 accelerate timm fastai==2.7.13 fastcore requests tqdm colorama
ERROR: Could not find a version that satisfies the requirement torch<2.2,>=1.10 (from fastai) (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch<2.2,>=1.10

[OK] apt-get
[OK] pip
Step 4/6 — Cloning DeOldify...
[OK] DeOldify already cloned.
Step 5/6 — Creating output directories...
Step 6/6 — Downloading DeOldify Artistic model weights...
[OK] Model weights already present.

✅ All dependencies installed successfully!


In [ ]:
import transformers
print(transformers.__version__)

5.10.1


In [ ]:
import torch
import transformers
import fastai

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("FastAI:", fastai.__version__)

Torch: 2.5.1+cu124
Transformers: 5.10.1
FastAI: 1.0.61


In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.46.3

Found existing installation: transformers 5.10.1
Uninstalling transformers-5.10.1:
  Successfully uninstalled transformers-5.10.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 61.8 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.17.0
    Uninstalling huggingface_hub-1.17.0:
      Successfully uninstalled huggingface_hub-1.17.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]


In [ ]:
import torch
import transformers
import fastai

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("FastAI:", fastai.__version__)

Torch: 2.5.1+cu124
Transformers: 4.46.3
FastAI: 1.0.61


In [ ]:
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
)

print("✅ SegFormer import successful")

✅ SegFormer import successful


---
## SECTION 2 — Import Libraries

In [ ]:
# ============================================================
# SECTION 2 — Import Libraries
# ============================================================

import os
import sys
import time
import copy
import warnings
import traceback
import logging
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, List, Tuple, Any

# Suppress noisy warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)

# ── Numerical & Image ──────────────────────────────────────
import numpy as np
import cv2
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb
from scipy.ndimage import gaussian_filter
from skimage import color as skcolor

# ── PyTorch ────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

# ── HuggingFace Transformers (SegFormer) ───────────────────
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
)

# ── DeOldify path injection ────────────────────────────────
DEOLDIFY_ROOT = "/content/DeOldify"
if DEOLDIFY_ROOT not in sys.path:
    sys.path.insert(0, DEOLDIFY_ROOT)

# ── Gradio ─────────────────────────────────────────────────
import gradio as gr

# ── Misc ───────────────────────────────────────────────────
from tqdm.auto import tqdm
import requests
from io import BytesIO

print(f"✅ Libraries loaded.")
print(f"   PyTorch  : {torch.__version__}")
print(f"   Gradio   : {gr.__version__}")
print(f"   CUDA     : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

✅ Libraries loaded.
   PyTorch  : 2.5.1+cu124
   Gradio   : 5.50.0
   CUDA     : True (Tesla T4)


---
## SECTION 3 — Configuration and Settings

In [ ]:
# ============================================================
# SECTION 3 — Configuration and Settings
# ============================================================

# ── Device ─────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths ──────────────────────────────────────────────────
DEOLDIFY_ROOT   = Path("/content/DeOldify")
MODEL_DIR       = DEOLDIFY_ROOT / "models"
OUTPUT_DIR      = Path("/content/outputs")
UPLOAD_DIR      = Path("/content/uploads")
DEOLDIFY_MODEL  = MODEL_DIR / "ColorizeArtistic_gen.pth"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# ── DeOldify render factor ─────────────────────────────────
# Higher = better quality but slower. 35 is the sweet spot for Colab T4.
DEOLDIFY_RENDER_FACTOR = 35

# ── Segmentation model ─────────────────────────────────────
# SegFormer-B2 fine-tuned on ADE20K (150 classes)
SEG_MODEL_NAME = "nvidia/segformer-b2-finetuned-ade-512-512"

# ── Supported semantic regions and their ADE20K class IDs ──
# ADE20K label list reference: https://groups.csail.mit.edu/vision/datasets/ADE20K/
REGION_CLASS_MAP: Dict[str, List[int]] = {
    "Sky":       [2],               # sky
    "Grass":     [9],               # grass
    "Trees":     [4, 17, 66, 72],   # tree, plant, palm tree, pine tree
    "Water":     [21, 26, 60, 109], # water, sea, lake, river
    "Buildings": [0, 1, 25, 48],    # wall, building, house, skyscraper
    "Road":      [6, 11, 52, 53],   # road, sidewalk, path, runway
    "Mountains": [16, 29],          # mountain, hill
    "Plants":    [17, 66, 68, 73],  # plant, flower, bush, shrub
    "Sand":      [46],              # sand
    "Snow":      [132],             # snow
}

# ── Default color palette (RGB tuples, 0-255) ───────────────
DEFAULT_COLORS: Dict[str, Tuple[int, int, int]] = {
    "Sky":       (135, 206, 235),   # Sky Blue
    "Grass":     (124, 179,  66),   # Grass Green
    "Trees":     ( 34, 102,  34),   # Dark Forest Green
    "Water":     ( 64, 164, 223),   # Ocean Blue
    "Buildings": (139, 110,  90),   # Warm Brown
    "Road":      (128, 128, 128),   # Concrete Gray
    "Mountains": (119, 136, 153),   # Slate Blue-Gray
    "Plants":    ( 85, 160,  70),   # Mid Green
    "Sand":      (194, 178, 128),   # Sandy Tan
    "Snow":      (255, 255, 255),   # White
}

# Named colors available in the GUI dropdown
NAMED_COLORS: Dict[str, Tuple[int, int, int]] = {
    "Sky Blue":      (135, 206, 235),
    "Deep Blue":     ( 30,  80, 162),
    "Ocean Blue":    ( 64, 164, 223),
    "Grass Green":   (124, 179,  66),
    "Dark Green":    ( 34, 102,  34),
    "Forest Green":  ( 34,  85,  34),
    "Lime Green":    (150, 210,  50),
    "Yellow":        (255, 220,  50),
    "Golden":        (218, 165,  32),
    "Warm Brown":    (139, 110,  90),
    "Dark Brown":    ( 90,  60,  30),
    "Concrete Gray": (128, 128, 128),
    "Dark Gray":     ( 80,  80,  80),
    "Slate Gray":    (119, 136, 153),
    "Sandy Tan":     (194, 178, 128),
    "Terracotta":    (204, 102,  51),
    "White":         (255, 255, 255),
    "Sunset Orange": (255, 120,  60),
    "Purple Dusk":   (120,  60, 150),
    "No Change":     None,           # sentinel — skip this region
}

# ── Blending alpha (0.0 = fully DeOldify, 1.0 = fully user color) ──
DEFAULT_ALPHA = 0.68

# ── Mask morphology kernel size ────────────────────────────
MORPH_KERNEL = 5

# ── Mask feather (Gaussian blur sigma) ─────────────────────
FEATHER_SIGMA = 8

print("✅ Configuration loaded.")
print(f"   Device          : {DEVICE}")
print(f"   Regions         : {list(REGION_CLASS_MAP.keys())}")
print(f"   Seg model       : {SEG_MODEL_NAME}")
print(f"   Default alpha   : {DEFAULT_ALPHA}")

✅ Configuration loaded.
   Device          : cuda
   Regions         : ['Sky', 'Grass', 'Trees', 'Water', 'Buildings', 'Road', 'Mountains', 'Plants', 'Sand', 'Snow']
   Seg model       : nvidia/segformer-b2-finetuned-ade-512-512
   Default alpha   : 0.68


In [ ]:
import torch
print(torch.__version__)

2.11.0+cu128


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 20.5 MB/s  0:00:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 45.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 23.8 MB/s  0:00:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 110.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8

In [ ]:
import torch
print("Torch:", torch.__version__)

import fastai
print("FastAI:", fastai.__version__)

Torch: 2.11.0+cu128
FastAI: 1.0.56.dev0


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip install fastai==1.0.61

Found existing installation: torch 2.5.1
Uninstalling torch-2.5.1:
  Successfully uninstalled torch-2.5.1
Found existing installation: torchvision 0.20.1
Uninstalling torchvision-0.20.1:
  Successfully uninstalled torchvision-0.20.1
Found existing installation: torchaudio 2.5.1
Uninstalling torchaudio-2.5.1:
  Successfully uninstalled torchaudio-2.5.1
  Using cached torch-2.5.1-cp312-cp312-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.20.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached torchaudio-2.5.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.4 kB)
Using cached torch-2.5.1-cp312-cp312-manylinux1_x86_64.whl (906.4 MB)
Using cached torchvision-0.20.1-cp312-cp312-manylinux1_x86_64.whl (7.2 MB)
Using cached torchaudio-2.5.1-cp312-cp312-manylinux1_x86_64.whl (3.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchaudio]
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject

In [ ]:
import torch
import fastai

print("Torch:", torch.__version__)
print("FastAI:", fastai.__version__)

Torch: 2.5.1+cu124
FastAI: 1.0.61


In [ ]:
!pip show torch
!pip show fastai

Name: torch
Version: 2.5.1
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision
Name: fastai
Version: 1.0.61
Summary: fastai makes deep learning with PyTorch faster, more accurate, and easier
Home-page: https://github.com/fastai/fastai
Author: Jeremy Howard
Author-email: info@fast.ai
License: Apache Software License 2.0
Location: /usr/local/lib/python3.12/di

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip install -q fastai==1.0.61

In [ ]:
print("DEOLDIFY_ROOT =", DEOLDIFY_ROOT)

DEOLDIFY_ROOT = /content/DeOldify


In [ ]:
import os
print(os.path.exists(DEOLDIFY_ROOT))

True


In [ ]:
!ls -la /content

total 28
drwxr-xr-x  1 root root 4096 Jun 11 05:54  .
drwxr-xr-x  1 root root 4096 Jun 11 05:47  ..
-rw-r--r--  1 root root    0 Jun 11 06:16 '=4.0.0'
-rw-r--r--  1 root root    0 Jun 11 06:16 '=4.35.0'
drwxr-xr-x  4 root root 4096 Jun  4 13:39  .config
drwxr-xr-x 10 root root 4096 Jun 11 05:54  DeOldify
drwxr-xr-x  2 root root 4096 Jun 11 05:54  outputs
drwxr-xr-x  1 root root 4096 Jun  4 13:39  sample_data
drwxr-xr-x  2 root root 4096 Jun 11 05:54  uploads


In [ ]:
print(DEOLDIFY_ROOT)

/content/DeOldify


In [ ]:
import os

print("DEOLDIFY_ROOT:", DEOLDIFY_ROOT)
print("Exists:", os.path.exists(DEOLDIFY_ROOT))
print("Is directory:", os.path.isdir(DEOLDIFY_ROOT))

DEOLDIFY_ROOT: /content/DeOldify
Exists: True
Is directory: True


In [ ]:
!ls -la /content/DeOldify

total 436
drwxr-xr-x 10 root root  4096 Jun 11 05:54 .
drwxr-xr-x  1 root root  4096 Jun 11 05:54 ..
-rw-r--r--  1 root root  7379 Jun 11 05:54 ColorFIDBenchmarkArtistic.ipynb
-rw-r--r--  1 root root 15025 Jun 11 05:54 ColorizeTrainingArtistic.ipynb
-rw-r--r--  1 root root 15033 Jun 11 05:54 ColorizeTrainingStable.ipynb
-rw-r--r--  1 root root 18892 Jun 11 05:54 ColorizeTrainingStableLargeBatch.ipynb
-rw-r--r--  1 root root 12035 Jun 11 05:54 ColorizeTrainingVideo.ipynb
-rw-r--r--  1 root root 27053 Jun 11 05:54 ColorizeTrainingWandb.ipynb
drwxr-xr-x  3 root root  4096 Jun 11 06:00 deoldify
-rw-r--r--  1 root root   333 Jun 11 05:54 environment.yml
drwxr-xr-x 11 root root  4096 Jun 11 06:00 fastai
drwxr-xr-x  2 root root  4096 Jun 11 05:54 fid
drwxr-xr-x  8 root root  4096 Jun 11 05:54 .git
drwxr-xr-x  2 root root  4096 Jun 11 05:54 .github
-rw-r--r--  1 root root  1537 Jun 11 05:54 .gitignore
-rw-r--r--  1 root root 79684 Jun 11 05:54 ImageColorizerArtisticTests.ipynb
-rw-r--r--  1 ro

In [ ]:
!ls -la /content/DeOldify/models

total 249180
drwxr-xr-x  2 root root      4096 Jun 11 05:54 .
drwxr-xr-x 10 root root      4096 Jun 11 05:54 ..
-rw-r--r--  1 root root 255144681 Dec 18  2022 ColorizeArtistic_gen.pth
-rw-r--r--  1 root root         1 Jun 11 05:54 .gitkeep


---
## SECTION 4 — DeOldify Model Setup

In [ ]:
# ============================================================
# SECTION 4 — DeOldify Model Setup
# ============================================================

import os
import tempfile
import functools
import numpy as np
import torch

from pathlib import Path
from PIL import Image
from skimage import color as skcolor

torch.backends.cudnn.benchmark = True

try:
    torch.serialization.add_safe_globals([functools.partial])
except:
    pass


class DeOldifyColorizer:
    """
    Wrapper around DeOldify artistic colorizer.
    Falls back to LAB colorization if DeOldify cannot load.
    """

    def __init__(self, model_path=None, render_factor=35):
        self.model_path = model_path
        self.render_factor = render_factor
        self.model = None
        self.is_loaded = False

        self._load()

    # --------------------------------------------------------
    # Load DeOldify
    # --------------------------------------------------------
    def _load(self):
        try:
            print("[DeOldify] Loading artistic colorizer...")

            from deoldify.visualize import get_image_colorizer

            self.model = get_image_colorizer(
                artistic=True,
                root_folder=Path(DEOLDIFY_ROOT)
            )

            self.is_loaded = True

            print("[DeOldify] ✅ Model loaded successfully.")

        except Exception as exc:

            print(f"[DeOldify] ⚠️ Could not load official model: {exc}")
            print("[DeOldify] Falling back to LAB colorization.")

            self.model = None
            self.is_loaded = False

    # --------------------------------------------------------
    # Run DeOldify
    # --------------------------------------------------------
    def _run_deoldify(self, pil_image):

        with tempfile.NamedTemporaryFile(
            suffix=".jpg",
            delete=False
        ) as tmp:

            temp_path = tmp.name

        pil_image.convert("RGB").save(temp_path)

        try:

            result = self.model.get_transformed_image(
                path=temp_path,
                render_factor=self.render_factor,
                watermarked=False
            )

            return result.convert("RGB")

        finally:

            if os.path.exists(temp_path):
                os.remove(temp_path)

    # --------------------------------------------------------
    # LAB fallback
    # --------------------------------------------------------
    def _lab_colorize_fallback(self, pil_image):

        gray = np.array(
            pil_image.convert("L")
        ).astype(np.float32)

        L = gray / 255.0 * 100.0

        A = np.full_like(L, 5.0)
        B = np.full_like(L, 10.0)

        lab = np.stack(
            [L, A, B],
            axis=-1
        )

        rgb = skcolor.lab2rgb(lab)

        rgb = (
            rgb * 255
        ).clip(
            0,
            255
        ).astype(
            np.uint8
        )

        return Image.fromarray(rgb)

    # --------------------------------------------------------
    # Public API
    # --------------------------------------------------------
    def colorize(self, pil_image):

        if pil_image is None:
            raise ValueError("Input image is None.")

        pil_image = pil_image.convert("RGB")

        if self.is_loaded and self.model is not None:

            try:
                return self._run_deoldify(
                    pil_image
                )

            except Exception as exc:

                print(
                    f"[DeOldify] Runtime error: {exc}"
                )

                return self._lab_colorize_fallback(
                    pil_image
                )

        return self._lab_colorize_fallback(
            pil_image
        )


# ------------------------------------------------------------
# Create singleton
# ------------------------------------------------------------

print("Initializing DeOldify colorizer...")

deoldify_colorizer = DeOldifyColorizer(
    render_factor=35
)

print(
    f"DeOldify ready. Official weights loaded: "
    f"{deoldify_colorizer.is_loaded}"
)

Initializing DeOldify colorizer...
[DeOldify] Loading artistic colorizer...
[DeOldify] ⚠️ Could not load official model: dummy is not a valid directory.
[DeOldify] Falling back to LAB colorization.
DeOldify ready. Official weights loaded: False


---
## SECTION 5 — Semantic Region Detection Module

In [ ]:
# ============================================================
# SECTION 5 — Semantic Region Detection Module
# ============================================================
# Uses SegFormer-B2 (nvidia/segformer-b2-finetuned-ade-512-512)
# to produce per-pixel semantic labels, then groups them into
# the 10 user-facing regions defined in REGION_CLASS_MAP.

class SemanticSegmenter:
    """
    Wraps SegFormer for semantic segmentation.
    Returns binary masks for each supported region.
    """

    def __init__(self, model_name: str = SEG_MODEL_NAME, device: torch.device = DEVICE):
        self.model_name = model_name
        self.device     = device
        self.processor  = None
        self.model      = None
        self._load()

    # ── Model loading ──────────────────────────────────────
    def _load(self):
        print(f"[Segmenter] Loading {self.model_name}...")
        try:
            self.processor = SegformerImageProcessor.from_pretrained(self.model_name)
            self.model     = SegformerForSemanticSegmentation.from_pretrained(self.model_name)
            self.model.to(self.device).eval()
            print(f"[Segmenter] ✅ Model loaded on {self.device}.")
        except Exception as exc:
            print(f"[Segmenter] ❌ Failed to load model: {exc}")
            raise

    # ── Inference ──────────────────────────────────────────
    @torch.no_grad()
    def segment(self, pil_image: Image.Image) -> np.ndarray:
        """
        Run segmentation on a PIL Image.
        Returns a (H, W) integer array of ADE20K class IDs.
        """
        if pil_image is None:
            raise ValueError("Input image is None.")

        rgb_image = pil_image.convert("RGB")
        orig_w, orig_h = rgb_image.size

        # Preprocess
        inputs = self.processor(
            images=rgb_image,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Inference
        outputs = self.model(**inputs)
        logits  = outputs.logits  # (1, num_classes, H/4, W/4)

        # Upsample to original resolution
        upsampled = F.interpolate(
            logits,
            size=(orig_h, orig_w),
            mode="bilinear",
            align_corners=False,
        )
        seg_map = upsampled.argmax(dim=1).squeeze().cpu().numpy().astype(np.int32)
        return seg_map

    # ── Region mask extraction ─────────────────────────────
    def get_region_masks(
        self,
        pil_image: Image.Image,
        morph_kernel: int = MORPH_KERNEL,
    ) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        """
        Segment image and return:
          - seg_map  : (H, W) raw label array
          - masks    : dict{region_name -> (H, W) uint8 binary mask (0 or 255)}
        Applies morphological cleaning to reduce noise.
        """
        seg_map = self.segment(pil_image)
        h, w    = seg_map.shape
        masks   = {}
        kernel  = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (morph_kernel, morph_kernel)
        )

        for region_name, class_ids in REGION_CLASS_MAP.items():
            # Build binary mask for all class IDs of this region
            raw_mask = np.zeros((h, w), dtype=np.uint8)
            for cid in class_ids:
                raw_mask[seg_map == cid] = 255

            if raw_mask.max() == 0:
                masks[region_name] = raw_mask  # empty mask
                continue

            # Morphological opening (removes speckle) then closing (fills gaps)
            cleaned = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN,  kernel)
            cleaned = cv2.morphologyEx(cleaned,  cv2.MORPH_CLOSE, kernel)
            masks[region_name] = cleaned

        return seg_map, masks

    # ── Segmentation visualization ─────────────────────────
    def visualize_masks(
        self,
        pil_image: Image.Image,
        masks: Dict[str, np.ndarray],
    ) -> Image.Image:
        """
        Create a color-coded overlay of all detected regions
        on top of the original image.
        Returns a PIL RGB Image.
        """
        base   = np.array(pil_image.convert("RGB"), dtype=np.float32)
        h, w   = base.shape[:2]
        overlay = base.copy()

        # Assign a vivid color to each region for the viz
        VIZ_COLORS = {
            "Sky":       (100, 180, 255),
            "Grass":     ( 80, 200,  80),
            "Trees":     ( 20, 120,  20),
            "Water":     ( 30, 120, 220),
            "Buildings": (200, 150, 100),
            "Road":      (180, 180, 180),
            "Mountains": (140, 160, 180),
            "Plants":    (120, 200,  80),
            "Sand":      (220, 195, 140),
            "Snow":      (240, 240, 255),
        }

        legend_entries = []
        for region, mask in masks.items():
            if mask.max() == 0:
                continue  # region not detected
            color = VIZ_COLORS.get(region, (200, 200, 200))
            bool_mask = mask > 0
            overlay[bool_mask] = (
                0.45 * overlay[bool_mask] +
                0.55 * np.array(color, dtype=np.float32)
            )
            legend_entries.append((region, color))

        # Draw legend using matplotlib
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        ax.imshow(overlay.astype(np.uint8))
        ax.axis("off")
        ax.set_title("Semantic Region Detection", fontsize=14, fontweight="bold")

        patches = [
            mpatches.Patch(
                facecolor=tuple(c / 255 for c in col),
                label=name,
            )
            for name, col in legend_entries
        ]
        if patches:
            ax.legend(
                handles=patches,
                loc="upper right",
                fontsize=8,
                framealpha=0.8,
            )

        fig.tight_layout(pad=0)
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf).convert("RGB")


# ── Instantiate segmenter (loaded once, reused across calls) ─
print("Initializing semantic segmenter...")
segmenter = SemanticSegmenter(model_name=SEG_MODEL_NAME, device=DEVICE)
print("Segmenter ready.")

Initializing semantic segmenter...
[Segmenter] Loading nvidia/segformer-b2-finetuned-ade-512-512...


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

[Segmenter] ✅ Model loaded on cuda.
Segmenter ready.


---
## SECTION 6 — Conditional Colorization Engine

In [ ]:
# ============================================================
# SECTION 6 — Conditional Colorization Engine
# ============================================================
# Given:
#   - deoldify_result  : PIL RGB image from DeOldify
#   - region_masks     : dict{region -> binary mask}
#   - user_conditions  : dict{region -> (R,G,B) or None}
#   - alpha            : blending weight
#
# Produces a final image where each masked region is blended
# with the user-specified color in LAB space (preserves luminance).

class ConditionalColorizationEngine:
    """
    Applies user-defined color conditions to semantic regions
    of a DeOldify-colorized image.

    Color transfer is performed in LAB space:
      - L channel is preserved (keeps texture & brightness)
      - A/B channels are blended toward the target color
    """

    def __init__(self, feather_sigma: float = FEATHER_SIGMA):
        self.feather_sigma = feather_sigma

    # ── Core color transfer ────────────────────────────────
    def _transfer_color_lab(
        self,
        base_rgb: np.ndarray,
        target_rgb: Tuple[int, int, int],
        mask: np.ndarray,
        alpha: float,
    ) -> np.ndarray:
        """
        Transfer `target_rgb` color into `base_rgb` wherever `mask > 0`.
        All blending is done in LAB to preserve luminance.

        Args:
            base_rgb   : (H, W, 3) uint8 RGB image (DeOldify output)
            target_rgb : (R, G, B) tuple 0-255
            mask       : (H, W) uint8 binary mask (0 or 255)
            alpha      : blend weight — 0.0 = keep base, 1.0 = full target

        Returns:
            (H, W, 3) uint8 RGB image with color applied in masked region
        """
        # Convert to float LAB
        base_lab   = skcolor.rgb2lab(base_rgb.astype(np.float32) / 255.0)
        result_lab = base_lab.copy()

        # Create a solid LAB image of the target color
        target_norm  = np.array(target_rgb, dtype=np.float32) / 255.0
        target_lab   = skcolor.rgb2lab(target_norm.reshape(1, 1, 3))[0, 0]  # (3,)

        # Feathered mask for smooth transitions (avoids hard edges)
        soft_mask = gaussian_filter(
            mask.astype(np.float32) / 255.0,
            sigma=self.feather_sigma,
        )  # values 0-1

        # Blend A and B channels; preserve L
        weight = soft_mask * alpha  # per-pixel blend weight
        result_lab[..., 1] = (
            (1 - weight) * base_lab[..., 1] +
            weight       * target_lab[1]
        )
        result_lab[..., 2] = (
            (1 - weight) * base_lab[..., 2] +
            weight       * target_lab[2]
        )

        # Back to RGB
        result_rgb = skcolor.lab2rgb(result_lab)
        result_rgb = (result_rgb * 255.0).clip(0, 255).astype(np.uint8)
        return result_rgb

    # ── Main apply method ──────────────────────────────────
    def apply(
        self,
        deoldify_image: Image.Image,
        region_masks:   Dict[str, np.ndarray],
        user_conditions: Dict[str, Optional[Tuple[int, int, int]]],
        alpha: float = DEFAULT_ALPHA,
    ) -> Tuple[Image.Image, Dict[str, Any]]:
        """
        Apply all user conditions and return:
          - result_image  : PIL RGB Image
          - stats         : dict of processing statistics

        Args:
            deoldify_image  : Base colorized image from DeOldify
            region_masks    : Per-region binary masks from segmenter
            user_conditions : {region_name: (R,G,B) or None}
            alpha           : Global blending strength (0-1)
        """
        if deoldify_image is None:
            raise ValueError("DeOldify image is None.")

        base_rgb = np.array(deoldify_image.convert("RGB"))
        result   = base_rgb.copy()
        stats    = {
            "applied_conditions": [],
            "skipped_regions":    [],
            "region_coverage":    {},
        }

        total_pixels = base_rgb.shape[0] * base_rgb.shape[1]

        for region_name, target_color in user_conditions.items():
            # Skip if user chose "No Change" (None)
            if target_color is None:
                stats["skipped_regions"].append(region_name)
                continue

            mask = region_masks.get(region_name)
            if mask is None or mask.max() == 0:
                stats["skipped_regions"].append(f"{region_name} (not detected)")
                continue

            # Coverage percentage
            coverage = float((mask > 0).sum()) / total_pixels * 100
            stats["region_coverage"][region_name] = round(coverage, 2)

            try:
                result = self._transfer_color_lab(
                    base_rgb=result,
                    target_rgb=target_color,
                    mask=mask,
                    alpha=alpha,
                )
                stats["applied_conditions"].append({
                    "region": region_name,
                    "color":  target_color,
                    "coverage_pct": round(coverage, 2),
                })
            except Exception as exc:
                print(f"[Engine] Error applying {region_name}: {exc}")
                stats["skipped_regions"].append(f"{region_name} (error: {exc})")

        return Image.fromarray(result, "RGB"), stats


# ── Instantiate engine ─────────────────────────────────────
engine = ConditionalColorizationEngine(feather_sigma=FEATHER_SIGMA)
print("✅ Conditional Colorization Engine ready.")

✅ Conditional Colorization Engine ready.


---
## SECTION 7 — Image Processing Pipeline

In [ ]:
# ============================================================
# SECTION 7 — Image Processing Pipeline
# ============================================================
# Orchestrates the full flow:
#   1. Validate & normalize input image
#   2. Run DeOldify colorization
#   3. Run semantic segmentation
#   4. Apply conditional colorization
#   5. Build 4-panel comparison figure
#   6. Collect statistics
#   7. Save outputs

class ImageProcessingPipeline:
    """
    End-to-end pipeline for conditional image colorization.
    """

    MAX_DIM = 1024  # Resize if larger (Colab RAM guard)

    def __init__(
        self,
        colorizer: DeOldifyColorizer,
        segmenter: SemanticSegmenter,
        cc_engine: ConditionalColorizationEngine,
        output_dir: Path = OUTPUT_DIR,
    ):
        self.colorizer  = colorizer
        self.segmenter  = segmenter
        self.cc_engine  = cc_engine
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)

    # ── Input validation ───────────────────────────────────
    def _validate_image(self, img_input) -> Image.Image:
        """
        Accept numpy array, PIL Image, or file path.
        Returns a PIL RGB Image, raises ValueError on failure.
        """
        if img_input is None:
            raise ValueError("No image provided.")

        if isinstance(img_input, np.ndarray):
            if img_input.ndim == 2:
                img = Image.fromarray(img_input, "L")
            elif img_input.shape[2] == 4:
                img = Image.fromarray(img_input, "RGBA").convert("RGB")
            else:
                img = Image.fromarray(img_input, "RGB")
        elif isinstance(img_input, Image.Image):
            img = img_input
        elif isinstance(img_input, (str, Path)):
            if not os.path.exists(img_input):
                raise FileNotFoundError(f"Image file not found: {img_input}")
            img = Image.open(img_input)
        else:
            raise TypeError(f"Unsupported image type: {type(img_input)}")

        img = img.convert("RGB")

        # Resize if too large
        w, h = img.size
        if max(w, h) > self.MAX_DIM:
            scale = self.MAX_DIM / max(w, h)
            new_w, new_h = int(w * scale), int(h * scale)
            img = img.resize((new_w, new_h), Image.LANCZOS)
            print(f"[Pipeline] Image resized: ({w},{h}) → ({new_w},{new_h})")

        if img.size[0] < 32 or img.size[1] < 32:
            raise ValueError("Image too small (minimum 32×32 pixels).")

        return img

    # ── 4-panel comparison figure ──────────────────────────
    def _build_comparison(
        self,
        grayscale:       Image.Image,
        deoldify_result: Image.Image,
        conditional:     Image.Image,
        seg_viz:         Image.Image,
    ) -> Image.Image:
        """
        Returns a 2×2 grid PIL image showing all four outputs.
        """
        titles = [
            "Original Grayscale",
            "DeOldify Colorized",
            "Conditional Colorized",
            "Region Segmentation",
        ]
        images = [grayscale, deoldify_result, conditional, seg_viz]

        fig, axes = plt.subplots(2, 2, figsize=(14, 11))
        fig.patch.set_facecolor("#1a1a2e")

        for ax, img, title in zip(axes.flatten(), images, titles):
            ax.imshow(img)
            ax.set_title(
                title,
                fontsize=13,
                fontweight="bold",
                color="white",
                pad=8,
            )
            ax.axis("off")

        fig.suptitle(
            "Conditional Image Colorization — Comparison",
            fontsize=16,
            fontweight="bold",
            color="white",
            y=1.01,
        )
        plt.tight_layout(pad=1.5)

        buf = BytesIO()
        fig.savefig(
            buf, format="png", dpi=100,
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
        )
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

    # ── Statistics formatting ──────────────────────────────
    def _format_stats(
        self,
        region_masks:      Dict[str, np.ndarray],
        user_conditions:   Dict[str, Optional[Tuple]],
        engine_stats:      Dict[str, Any],
        timing:            Dict[str, float],
        deoldify_loaded:   bool,
    ) -> str:
        """
        Build a human-readable statistics Markdown string.
        """
        detected = [
            name for name, mask in region_masks.items()
            if mask.max() > 0
        ]

        lines = [
            "## 📊 Processing Statistics",
            "",
            "### 🔍 Detected Regions",
        ]
        if detected:
            for name in detected:
                pct = engine_stats["region_coverage"].get(name, 0)
                lines.append(f"- **{name}** — {pct:.1f}% coverage")
        else:
            lines.append("- No regions detected.")

        lines += [
            "",
            "### 🎨 Applied Conditions",
        ]
        applied = engine_stats.get("applied_conditions", [])
        if applied:
            for cond in applied:
                r, g, b = cond["color"]
                lines.append(
                    f"- **{cond['region']}** → RGB({r},{g},{b}) "
                    f"| {cond['coverage_pct']:.1f}% of image"
                )
        else:
            lines.append("- No conditions applied.")

        skipped = engine_stats.get("skipped_regions", [])
        if skipped:
            lines += ["", "### ⏩ Skipped"]
            for s in skipped:
                lines.append(f"- {s}")

        lines += [
            "",
            "### ⏱️ Timing",
            f"- DeOldify colorization : `{timing.get('deoldify', 0):.2f}s`",
            f"- Semantic segmentation : `{timing.get('segmentation', 0):.2f}s`",
            f"- Conditional colorize  : `{timing.get('conditional', 0):.2f}s`",
            f"- **Total**             : `{timing.get('total', 0):.2f}s`",
            "",
            "### ℹ️ System",
            f"- DeOldify official weights : `{'✅ Loaded' if deoldify_loaded else '⚠️ Fallback (LAB stub)'}`",
            f"- Device                    : `{DEVICE}`",
        ]

        return "\n".join(lines)

    # ── Main run ───────────────────────────────────────────
    def run(
        self,
        img_input,
        user_conditions: Dict[str, Optional[Tuple[int, int, int]]],
        alpha: float = DEFAULT_ALPHA,
    ) -> Tuple[
        Image.Image,   # grayscale
        Image.Image,   # deoldify
        Image.Image,   # conditional
        Image.Image,   # seg_viz
        Image.Image,   # comparison panel
        str,           # stats text
        str,           # download path
    ]:
        """
        Full pipeline run.
        Returns (grayscale, deoldify, conditional, seg_viz, comparison, stats, save_path).
        Raises ValueError / RuntimeError on failures.
        """
        t_start = time.time()
        timing  = {}

        # 1. Validate input
        input_image = self._validate_image(img_input)
        grayscale   = input_image.convert("L").convert("RGB")  # display-friendly

        # 2. DeOldify colorization
        t0 = time.time()
        try:
            deoldify_result = self.colorizer.colorize(input_image)
        except Exception as exc:
            raise RuntimeError(f"DeOldify failed: {exc}") from exc
        timing["deoldify"] = time.time() - t0

        # 3. Semantic segmentation
        t0 = time.time()
        try:
            seg_map, region_masks = self.segmenter.get_region_masks(input_image)
            seg_viz = self.segmenter.visualize_masks(input_image, region_masks)
        except Exception as exc:
            raise RuntimeError(f"Segmentation failed: {exc}") from exc
        timing["segmentation"] = time.time() - t0

        # 4. Conditional colorization
        t0 = time.time()
        try:
            conditional_result, engine_stats = self.cc_engine.apply(
                deoldify_image=deoldify_result,
                region_masks=region_masks,
                user_conditions=user_conditions,
                alpha=alpha,
            )
        except Exception as exc:
            raise RuntimeError(f"Conditional colorization failed: {exc}") from exc
        timing["conditional"] = time.time() - t0

        timing["total"] = time.time() - t_start

        # 5. Comparison panel
        comparison = self._build_comparison(
            grayscale, deoldify_result, conditional_result, seg_viz
        )

        # 6. Statistics
        stats_text = self._format_stats(
            region_masks=region_masks,
            user_conditions=user_conditions,
            engine_stats=engine_stats,
            timing=timing,
            deoldify_loaded=self.colorizer.is_loaded,
        )

        # 7. Save outputs
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = str(self.output_dir / f"conditional_colorized_{ts}.png")
        conditional_result.save(save_path, format="PNG")

        # Also save comparison
        comparison.save(
            str(self.output_dir / f"comparison_{ts}.png"), format="PNG"
        )

        print(f"[Pipeline] ✅ Done in {timing['total']:.2f}s. Saved to {save_path}")
        return (
            grayscale,
            deoldify_result,
            conditional_result,
            seg_viz,
            comparison,
            stats_text,
            save_path,
        )


# ── Instantiate pipeline ────────────────────────────────────
pipeline = ImageProcessingPipeline(
    colorizer=deoldify_colorizer,
    segmenter=segmenter,
    cc_engine=engine,
    output_dir=OUTPUT_DIR,
)
print("✅ Image Processing Pipeline ready.")

✅ Image Processing Pipeline ready.


---
## SECTION 8 — Gradio GUI

In [ ]:
# ============================================================
# SECTION 8 — Gradio GUI
# ============================================================
# Full interactive interface with:
#   • Image upload (grayscale or RGB)
#   • Per-region color dropdowns
#   • Blending strength slider
#   • Render factor slider (DeOldify quality)
#   • 4-panel output display
#   • Statistics panel
#   • Download button for final result

COLOR_NAMES = list(NAMED_COLORS.keys())  # dropdown options


def _parse_conditions(
    sky_color:       str,
    grass_color:     str,
    trees_color:     str,
    water_color:     str,
    buildings_color: str,
    road_color:      str,
    mountains_color: str,
    plants_color:    str,
    sand_color:      str,
    snow_color:      str,
) -> Dict[str, Optional[Tuple[int, int, int]]]:
    """
    Convert GUI dropdown values to {region: (R,G,B) or None} dict.
    'No Change' → None (skip this region).
    """
    raw = {
        "Sky":       sky_color,
        "Grass":     grass_color,
        "Trees":     trees_color,
        "Water":     water_color,
        "Buildings": buildings_color,
        "Road":      road_color,
        "Mountains": mountains_color,
        "Plants":    plants_color,
        "Sand":      sand_color,
        "Snow":      snow_color,
    }
    return {region: NAMED_COLORS[name] for region, name in raw.items()}


def gradio_run(
    image,
    sky_color, grass_color, trees_color, water_color,
    buildings_color, road_color, mountains_color,
    plants_color, sand_color, snow_color,
    alpha_slider,
    render_factor_slider,
):
    """
    Main Gradio callback.
    Returns (grayscale, deoldify, conditional, seg_viz, comparison, stats_md, download_path).
    All errors are caught and surfaced as a stats message.
    """
    # ── Input guard ────────────────────────────────────────
    if image is None:
        err = "⚠️ Please upload an image first."
        return None, None, None, None, None, err, None

    try:
        # Update render factor on-the-fly
        pipeline.colorizer.render_factor = int(render_factor_slider)

        # Parse color conditions from GUI
        user_conditions = _parse_conditions(
            sky_color, grass_color, trees_color, water_color,
            buildings_color, road_color, mountains_color,
            plants_color, sand_color, snow_color,
        )

        # Run pipeline
        (
            grayscale,
            deoldify_result,
            conditional_result,
            seg_viz,
            comparison,
            stats_text,
            save_path,
        ) = pipeline.run(
            img_input=image,
            user_conditions=user_conditions,
            alpha=float(alpha_slider),
        )

        return (
            np.array(grayscale),
            np.array(deoldify_result),
            np.array(conditional_result),
            np.array(seg_viz),
            np.array(comparison),
            stats_text,
            save_path,
        )

    except ValueError as exc:
        msg = f"## ❌ Input Error\n\n`{exc}`"
        return None, None, None, None, None, msg, None
    except FileNotFoundError as exc:
        msg = f"## ❌ File Not Found\n\n`{exc}`"
        return None, None, None, None, None, msg, None
    except RuntimeError as exc:
        msg = f"## ❌ Processing Error\n\n`{exc}`"
        print(traceback.format_exc())
        return None, None, None, None, None, msg, None
    except Exception as exc:
        msg = f"## ❌ Unexpected Error\n\n`{exc}`\n\nCheck the Colab console for details."
        print(traceback.format_exc())
        return None, None, None, None, None, msg, None


# ── Build default condition values for the GUI ─────────────
def _get_default_color_name(region: str) -> str:
    """Return the named color whose RGB matches the default for this region."""
    default_rgb = DEFAULT_COLORS.get(region)
    if default_rgb is None:
        return "No Change"
    for name, rgb in NAMED_COLORS.items():
        if rgb == default_rgb:
            return name
    return "No Change"


# ── CSS theme ──────────────────────────────────────────────
CUSTOM_CSS = """
#title-banner {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    padding: 24px 32px;
    border-radius: 12px;
    margin-bottom: 16px;
    text-align: center;
}
#title-banner h1 {
    color: #e0e0ff;
    font-size: 2em;
    font-weight: 700;
    margin: 0;
}
#title-banner p {
    color: #a0aec0;
    font-size: 0.95em;
    margin: 8px 0 0 0;
}
.section-header {
    font-weight: 700;
    color: #63b3ed;
    font-size: 1.05em;
    margin-top: 12px;
    border-bottom: 1px solid #2d3748;
    padding-bottom: 4px;
}
"""


# ── Build the Gradio Blocks interface ──────────────────────
with gr.Blocks(
    title="Conditional Image Colorization",
    theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate"),
    css=CUSTOM_CSS,
) as demo:

    # ── Title Banner ────────────────────────────────────────
    gr.HTML("""
    <div id="title-banner">
        <h1>🎨 Conditional Image Colorization</h1>
        <p>
            Colorize grayscale photos with DeOldify · Apply user-defined colors to semantic regions ·
            Powered by SegFormer-B2 + LAB color transfer
        </p>
    </div>
    """)

    with gr.Row():
        # ── LEFT COLUMN — Input & Controls ──────────────────
        with gr.Column(scale=1, min_width=320):
            gr.Markdown("### 📤 Upload Image")
            image_input = gr.Image(
                label="Upload Grayscale or Color Image",
                type="numpy",
                height=280,
            )

            gr.Markdown("### 🎛️ Model Settings")
            alpha_slider = gr.Slider(
                minimum=0.0, maximum=1.0, value=DEFAULT_ALPHA, step=0.05,
                label="Color Blending Strength",
                info="0 = keep DeOldify colors · 1 = full user color",
            )
            render_factor_slider = gr.Slider(
                minimum=10, maximum=45, value=DEOLDIFY_RENDER_FACTOR, step=1,
                label="DeOldify Render Factor",
                info="Higher = better quality, slower",
            )

            gr.Markdown("### 🌈 Color Conditions")
            gr.Markdown("*Select a color for each region, or 'No Change' to leave it untouched.*")

            sky_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Sky"),
                label="☁️  Sky",
            )
            grass_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Grass"),
                label="🌿  Grass",
            )
            trees_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Trees"),
                label="🌳  Trees",
            )
            water_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Water"),
                label="💧  Water",
            )
            buildings_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Buildings"),
                label="🏢  Buildings",
            )
            road_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Road"),
                label="🛣️  Road",
            )
            mountains_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Mountains"),
                label="⛰️  Mountains",
            )
            plants_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Plants"),
                label="🌱  Plants",
            )
            sand_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Sand"),
                label="🏖️  Sand",
            )
            snow_dd = gr.Dropdown(
                choices=COLOR_NAMES, value=_get_default_color_name("Snow"),
                label="❄️  Snow",
            )

            with gr.Row():
                run_btn = gr.Button(
                    "🚀 Colorize",
                    variant="primary",
                    size="lg",
                )
                clear_btn = gr.ClearButton(
                    value="🗑️ Clear",
                    size="lg",
                )

        # ── RIGHT COLUMN — Outputs ───────────────────────────
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Results")

            with gr.Tabs():
                with gr.Tab("🖼️ Comparison Panel"):
                    comparison_out = gr.Image(
                        label="4-Panel Comparison",
                        type="numpy",
                        height=520,
                    )

                with gr.Tab("🎨 Individual Outputs"):
                    with gr.Row():
                        gray_out = gr.Image(
                            label="Original Grayscale",
                            type="numpy",
                            height=280,
                        )
                        deoldify_out = gr.Image(
                            label="DeOldify Colorized",
                            type="numpy",
                            height=280,
                        )
                    with gr.Row():
                        conditional_out = gr.Image(
                            label="Conditional Colorized",
                            type="numpy",
                            height=280,
                        )
                        seg_viz_out = gr.Image(
                            label="Region Segmentation",
                            type="numpy",
                            height=280,
                        )

                with gr.Tab("📈 Statistics"):
                    stats_out = gr.Markdown(
                        value="*Run the pipeline to see statistics.*",
                    )

            # Download button
            gr.Markdown("### 💾 Download")
            download_btn = gr.File(
                label="Download Conditional Colorized Image (PNG)",
                file_count="single",
                interactive=False,
            )

    # ── Examples ────────────────────────────────────────────
    gr.Markdown("---")
    gr.Markdown("""
    ### 💡 How to Use
    1. **Upload** a grayscale (or color) photograph
    2. **Select** desired colors for each semantic region using the dropdowns
    3. **Adjust** Blending Strength (how strongly user colors override DeOldify's choices)
    4. Click **🚀 Colorize** — results appear in the tabs above
    5. **Download** the final conditional colorized image

    > **Tip:** Set regions you don't care about to **No Change** to keep DeOldify's automatic coloring for those areas.
    """)

    # ── Wire up callbacks ───────────────────────────────────
    run_inputs = [
        image_input,
        sky_dd, grass_dd, trees_dd, water_dd,
        buildings_dd, road_dd, mountains_dd,
        plants_dd, sand_dd, snow_dd,
        alpha_slider,
        render_factor_slider,
    ]
    run_outputs = [
        gray_out,
        deoldify_out,
        conditional_out,
        seg_viz_out,
        comparison_out,
        stats_out,
        download_btn,
    ]

    run_btn.click(
        fn=gradio_run,
        inputs=run_inputs,
        outputs=run_outputs,
        show_progress=True,
    )

print("✅ Gradio interface built successfully.")

✅ Gradio interface built successfully.


INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


---
## SECTION 9 — Output Saving and Download System

In [ ]:
# ============================================================
# SECTION 9 — Output Saving and Download System
# ============================================================
# Utility functions for listing, previewing, and downloading
# saved outputs — handy for Colab sessions.

def list_outputs(directory: Path = OUTPUT_DIR) -> List[str]:
    """Return a sorted list of all PNG files in the output directory."""
    files = sorted(directory.glob("*.png"), key=os.path.getmtime, reverse=True)
    if not files:
        print("No output files found yet.")
        return []
    print(f"📁 {len(files)} file(s) in {directory}:")
    for f in files:
        size_kb = os.path.getsize(f) / 1024
        mtime   = datetime.fromtimestamp(os.path.getmtime(f)).strftime("%Y-%m-%d %H:%M:%S")
        print(f"  {f.name:45s}  {size_kb:7.1f} KB  {mtime}")
    return [str(f) for f in files]


def preview_outputs(n: int = 3, directory: Path = OUTPUT_DIR):
    """
    Display the most recent `n` conditional-colorized outputs inline.
    Only shows files matching 'conditional_colorized_*.png'.
    """
    files = sorted(
        directory.glob("conditional_colorized_*.png"),
        key=os.path.getmtime,
        reverse=True,
    )[:n]

    if not files:
        print("No conditional colorized outputs found.")
        return

    fig, axes = plt.subplots(1, len(files), figsize=(6 * len(files), 5))
    if len(files) == 1:
        axes = [axes]

    for ax, f in zip(axes, files):
        img = Image.open(f)
        ax.imshow(img)
        ax.set_title(f.name, fontsize=8)
        ax.axis("off")

    plt.suptitle("Recent Conditional Colorization Outputs", fontweight="bold")
    plt.tight_layout()
    plt.show()


def download_latest(directory: Path = OUTPUT_DIR):
    """
    Trigger a Colab file download for the most recent conditional output.
    """
    try:
        from google.colab import files as colab_files
    except ImportError:
        print("Not running in Google Colab — use the Gradio download button instead.")
        return

    outputs = sorted(
        directory.glob("conditional_colorized_*.png"),
        key=os.path.getmtime,
        reverse=True,
    )
    if not outputs:
        print("No output file to download yet. Run the pipeline first.")
        return

    latest = str(outputs[0])
    print(f"Downloading: {latest}")
    colab_files.download(latest)


def clear_outputs(directory: Path = OUTPUT_DIR, keep_n: int = 5):
    """
    Keep only the most recent `keep_n` outputs and delete the rest.
    Useful for freeing Colab disk space during long sessions.
    """
    all_files = sorted(
        directory.glob("*.png"),
        key=os.path.getmtime,
        reverse=True,
    )
    to_delete = all_files[keep_n:]
    for f in to_delete:
        f.unlink()
    print(f"[OutputManager] Kept {min(keep_n, len(all_files))} files, deleted {len(to_delete)}.")


print("✅ Output saving and download utilities ready.")
print(f"   Output directory: {OUTPUT_DIR}")

✅ Output saving and download utilities ready.
   Output directory: /content/outputs


---
## SECTION 10 — Application Launch

In [ ]:
# ============================================================
# SECTION 10 — Application Launch
# ============================================================
# Launches the Gradio app.
# In Google Colab, share=True generates a public tunnel URL
# that is accessible from any browser for 72 hours.

print("="*60)
print(" Conditional Image Colorization — Launch")
print("="*60)
print(f" Device              : {DEVICE}")
print(f" DeOldify weights    : {'✅ Official' if deoldify_colorizer.is_loaded else '⚠️ LAB Fallback'}")
print(f" Segmenter           : SegFormer-B2 (ADE20K 150 classes)")
print(f" Output directory    : {OUTPUT_DIR}")
print(f" Supported regions   : {len(REGION_CLASS_MAP)}")
print("="*60)

# Optionally show any previously saved outputs
list_outputs()

# ── Launch ─────────────────────────────────────────────────
demo.launch(
    share=True,           # Generates public URL in Colab
    debug=False,          # Set True to see full tracebacks in Gradio logs
    show_error=True,      # Show errors inside the UI
    quiet=False,
    height=900,
)

 Conditional Image Colorization — Launch
 Device              : cuda
 DeOldify weights    : ⚠️ LAB Fallback
 Segmenter           : SegFormer-B2 (ADE20K 150 classes)
 Output directory    : /content/outputs
 Supported regions   : 10
No output files found yet.


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"


HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"


HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
* Running on public URL: https://c5053f323ba87bc848.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


INFO:httpx:HTTP Request: HEAD https://c5053f323ba87bc848.gradio.live "HTTP/1.1 200 OK"


HTTP Request: HEAD https://c5053f323ba87bc848.gradio.live "HTTP/1.1 200 OK"


---
## Optional — Batch Processing Utility
*Run this cell independently to colorize multiple images without the GUI.*

In [ ]:
# ============================================================
# OPTIONAL — Batch Processing
# ============================================================
# Process a list of images with the same color conditions
# programmatically (useful for automated evaluation).

def batch_colorize(
    image_paths: List[str],
    user_conditions: Optional[Dict[str, Optional[Tuple]]] = None,
    alpha: float = DEFAULT_ALPHA,
) -> List[str]:
    """
    Colorize a list of images and return paths to saved outputs.

    Args:
        image_paths     : list of file paths to grayscale images
        user_conditions : {region: (R,G,B) or None}; uses defaults if None
        alpha           : blending strength

    Returns:
        List of save paths for each conditional-colorized result.
    """
    if user_conditions is None:
        # Use all defaults
        user_conditions = {r: DEFAULT_COLORS[r] for r in REGION_CLASS_MAP}

    saved = []
    for i, img_path in enumerate(image_paths):
        print(f"\n[Batch] Processing {i+1}/{len(image_paths)}: {img_path}")
        try:
            _, _, _, _, _, _, save_path = pipeline.run(
                img_input=img_path,
                user_conditions=user_conditions,
                alpha=alpha,
            )
            saved.append(save_path)
        except Exception as exc:
            print(f"[Batch] ❌ Failed on {img_path}: {exc}")

    print(f"\n[Batch] ✅ Processed {len(saved)}/{len(image_paths)} images.")
    preview_outputs(n=min(3, len(saved)))
    return saved


# Example usage (uncomment to run):
# paths = ["/content/uploads/photo1.jpg", "/content/uploads/photo2.jpg"]
# results = batch_colorize(paths)

print("✅ Batch processing utility ready.")

---
## Optional — Download Latest Output
*Run after generating results to download the file directly.*

In [ ]:
# Run this cell to download the most recently generated output
download_latest()